# Lab 7: Dynamic Programming

Lab associated with Module 7: Dynamic Programming

***

In [1]:
# The following lines are used to increase the width of cells to utilize more space on the screen 
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

***

### Section 0: Imports

In [2]:
import numpy as np

In [3]:
import math

In [4]:
from IPython.display import Image
from graphviz import Digraph

Details of Digraph package: https://h1ros.github.io/posts/introduction-to-graphviz-in-jupyter-notebook/

***

### <font color='red'> Activity 1: You are running up a staircase with a total of n steps. You can hop either 1 step, 2 steps or 3 steps at at time. Write a DP program to determine how many possible ways you can run up the stairs? (Hint: Start with a recursive solution, and then later move to top-down approach of DP). </font>

In [5]:
import time
import sys

sys.setrecursionlimit(10000)


def stairs_recursive(n):
    """Plain recursion. Every call re-solves the same sub-problems."""
    if n < 0:
        return 0
    if n == 0:
        return 1          # one way to stand still at the top
    return stairs_recursive(n - 1) + stairs_recursive(n - 2) + stairs_recursive(n - 3)


def stairs_topdown(n, memo=None):
    """Top-down DP. Same recursion, but each n is only ever solved once."""
    if memo is None:
        memo = {}
    if n < 0:
        return 0
    if n == 0:
        return 1
    if n in memo:
        return memo[n]
    memo[n] = (stairs_topdown(n - 1, memo)
               + stairs_topdown(n - 2, memo)
               + stairs_topdown(n - 3, memo))
    return memo[n]


def stairs_bottomup(n):
    """Bottom-up DP. Fill the table from step 0 upwards, no recursion at all."""
    if n < 0:
        return 0
    ways = [0] * (n + 1)
    ways[0] = 1
    for i in range(1, n + 1):
        ways[i] = ways[i - 1]
        if i >= 2:
            ways[i] += ways[i - 2]
        if i >= 3:
            ways[i] += ways[i - 3]
    return ways[n]

In [6]:
# All three versions must agree
print('n   recursive  top-down  bottom-up')
for n in range(0, 11):
    print('{:<3} {:<10} {:<9} {}'.format(n,
                                         stairs_recursive(n),
                                         stairs_topdown(n),
                                         stairs_bottomup(n)))

n   recursive  top-down  bottom-up
0   1          1         1
1   1          1         1
2   2          2         2
3   4          4         4
4   7          7         7
5   13         13        13
6   24         24        24
7   44         44        44
8   81         81        81
9   149        149       149
10  274        274       274


In [7]:
# Timing. The recursive version blows up, the two DP versions do not.
print('n    recursive(ms)  top-down(ms)  bottom-up(ms)   ways')
for n in [20, 25, 30]:
    t0 = time.perf_counter(); ways = stairs_recursive(n)
    t1 = time.perf_counter(); stairs_topdown(n)
    t2 = time.perf_counter(); stairs_bottomup(n)
    t3 = time.perf_counter()
    print('{:<4} {:<14.3f} {:<13.4f} {:<15.4f} {}'.format(
        n, (t1 - t0) * 1000, (t2 - t1) * 1000, (t3 - t2) * 1000, ways))

n    recursive(ms)  top-down(ms)  bottom-up(ms)   ways
20   20.770         0.0124        0.0085          121415


25   440.173        0.0212        0.0112          2555757


30   9363.610       0.0220        0.0125          53798080


In [8]:
# Only the DP versions can handle a large n at all
print('n = 500 has a {} digit answer'.format(len(str(stairs_bottomup(500)))))
print('top-down agrees with bottom-up:', stairs_topdown(500) == stairs_bottomup(500))

n = 500 has a 133 digit answer
top-down agrees with bottom-up: True


### <font color='red'> Activity 2: Write the code for finding the Longest Common Sub-sequence. Make sure you output the Matrix C and teh longest sub-sequence as well. Test your code with various use-cases. </font>

In [9]:
def LCS(X, Y):
    """Classic O(mn) LCS. Returns the length, the sub-sequence itself and the matrix C."""
    m, n = len(X), len(Y)

    # C[i][j] is the length of the LCS of the prefixes X_i and Y_j
    C = np.zeros((m + 1, n + 1), dtype=int)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if X[i - 1] == Y[j - 1]:
                # characters match, so they extend the LCS of the two shorter prefixes
                C[i][j] = C[i - 1][j - 1] + 1
            else:
                # no match, so drop the last character of X or of Y, whichever is better
                C[i][j] = max(C[i - 1][j], C[i][j - 1])

    return C[m][n], backtrack(X, Y, C), C


def backtrack(X, Y, C):
    """Walk back from C[m][n] to C[0][0] to recover one longest common sub-sequence."""
    i, j = len(X), len(Y)
    letters = []
    while i > 0 and j > 0:
        if X[i - 1] == Y[j - 1]:
            letters.append(X[i - 1])      # this character is in the LCS, move diagonally
            i -= 1
            j -= 1
        elif C[i - 1][j] >= C[i][j - 1]:
            i -= 1                        # value came from above
        else:
            j -= 1                        # value came from the left
    letters.reverse()
    return ''.join(letters)


def print_matrix(X, Y, C):
    """Print C with the two sequences as row and column headings."""
    print('       ' + '  '.join('{:>2}'.format(c) for c in ' ' + Y))
    for i in range(len(X) + 1):
        label = ' ' if i == 0 else X[i - 1]
        print(' {} '.format(label) + '  '.join('{:>2}'.format(v) for v in C[i]))

In [10]:
# Worked example from the notes
X, Y = 'ABCDEFGH', 'ABDFGHI'
length, seq, C = LCS(X, Y)

print_matrix(X, Y, C)
print('\nLCS length:', length)
print('LCS:', seq)

            A   B   D   F   G   H   I
    0   0   0   0   0   0   0   0
 A  0   1   1   1   1   1   1   1
 B  0   1   2   2   2   2   2   2
 C  0   1   2   2   2   2   2   2
 D  0   1   2   3   3   3   3   3
 E  0   1   2   3   3   3   3   3
 F  0   1   2   3   4   4   4   4
 G  0   1   2   3   4   5   5   5
 H  0   1   2   3   4   5   6   6

LCS length: 6
LCS: ABDFGH


In [11]:
# The two frog DNA sequences from the module
X = 'AGCCCTAAGGGCTACCTAGCTT'
Y = 'GACAGCCTACAAGCGTTAGCTTG'
length, seq, C = LCS(X, Y)

print('LCS length:', length)
print('LCS:', seq)

# The notes quote AGCCTAAGCTTAGCTT, which is also 16 long. An LCS is not unique,
# so my backtracking returns a different but equally long answer.


def is_subsequence(s, t):
    it = iter(t)
    return all(c in it for c in s)


print('mine is a sub-sequence of both:', is_subsequence(seq, X) and is_subsequence(seq, Y))
print('theirs is a sub-sequence of both:',
      is_subsequence('AGCCTAAGCTTAGCTT', X) and is_subsequence('AGCCTAAGCTTAGCTT', Y))

LCS length: 16
LCS: AGCCCAAGGTTAGCTT
mine is a sub-sequence of both: True
theirs is a sub-sequence of both: True


In [12]:
# Edge cases
tests = [
    ('', ''),                 # both empty
    ('', 'ABC'),              # one empty
    ('ABC', ''),              # the other empty
    ('ABC', 'XYZ'),           # nothing in common
    ('AAAA', 'AA'),           # repeated characters
    ('ABCD', 'ABCD'),         # identical
    ('ABCD', 'DCBA'),         # reversed
    ('AGGTAB', 'GXTXAYB'),    # standard textbook pair
]

for X, Y in tests:
    length, seq, _ = LCS(X, Y)
    print("X={:<10} Y={:<10} length={}  LCS='{}'".format(repr(X), repr(Y), length, seq))

X=''         Y=''         length=0  LCS=''
X=''         Y='ABC'      length=0  LCS=''
X='ABC'      Y=''         length=0  LCS=''
X='ABC'      Y='XYZ'      length=0  LCS=''
X='AAAA'     Y='AA'       length=2  LCS='AA'
X='ABCD'     Y='ABCD'     length=4  LCS='ABCD'
X='ABCD'     Y='DCBA'     length=1  LCS='A'
X='AGGTAB'   Y='GXTXAYB'  length=4  LCS='GTAB'


***

### Section 2: Unbounded Knapsack Problem

Let us build a solution to unbounded Knapsack problem.

In [13]:
def unboundedKnapsack(W, n, wt, vals, names):
 
    K = [0 for i in range(W + 1)]
    ITEMS = [[] for i in range(W + 1)]
 
    for x in range(1, W + 1):
        K[x] = 0
        for i in range(1, n):
            
            prev_k = K[x]
            
            if (wt[i] <= x):
                K[x] = max(K[x], K[x - wt[i]] + vals[i])
                
            if K[x] != prev_k:
                ITEMS[x] = ITEMS[x - wt[i]] + names[i]
                
 
    return K[W], ITEMS[W]

In [14]:
W = 4
wt = [1, 2, 3]
vals = [1, 4, 6]
names = [["Turtle"], ["Globe"], ["WaterMelon"]]

n = len(vals)   # the template said len(V), but V is not defined until the 0/1 section

print('We have {} items'.format(n))

We have 3 items


In [15]:
K, ITEMS = unboundedKnapsack(W, n, wt, vals, names)

In [16]:
ITEMS

['Globe', 'Globe']

***

### <font color='red'> Activity 3: In the earlier activity, you analysed the code for unbounded knapsack. Based on the algorithm discussed in this section, implement a solution to do 0/1 Knapsack. Make sure you test your algorithms for various test-cases. </font>

In [17]:
def knapsack01(W, n, wt, vals, names):
    """0/1 knapsack. Each item may be used at most once.

    Changes from the unbounded version:
      1. K is two dimensional, K[j][x], instead of one dimensional K[x].
         The extra dimension j tracks how many items are available, which is what
         stops an item being reused.
      2. When we take item j we look back at K[j - 1][x - wt[j - 1]], the row
         *without* item j. The unbounded version looked back at K[x - wt[i]] on the
         same row, which is exactly what let it pick the same item again.
      3. The loop runs j from 1 to n inclusive so no item is skipped. The template
         unbounded code used range(1, n), which silently ignores item 0.
      4. The chosen items are recovered afterwards by walking back through K rather
         than being built up during the fill.
    """
    # K[j][x] = best value using only the first j items, with capacity x
    K = [[0 for x in range(W + 1)] for j in range(n + 1)]

    for j in range(1, n + 1):
        for x in range(0, W + 1):
            # option 1: leave item j out
            K[j][x] = K[j - 1][x]

            # option 2: take item j, if it fits, and add it to the best solution
            # built from the earlier items only
            if wt[j - 1] <= x:
                K[j][x] = max(K[j][x], K[j - 1][x - wt[j - 1]] + vals[j - 1])

    return K[n][W], recover_items(W, n, wt, names, K), K


def recover_items(W, n, wt, names, K):
    """Walk back through K. If a cell differs from the row above, item j was taken."""
    chosen = []
    x = W
    for j in range(n, 0, -1):
        if K[j][x] != K[j - 1][x]:
            chosen.append(names[j - 1])
            x -= wt[j - 1]
    chosen.reverse()
    return chosen

Class Room Test-case

In [18]:
W = 10
V = [20, 8, 14, 13, 35]
w = [6, 2, 4, 3, 11]
item_names = ['item1', 'item2', 'item3', 'item4', 'item5']

n = len(V)

print('We have {} items'.format(n))

We have 5 items


In [19]:
best, chosen, K = knapsack01(W, n, w, V, item_names)

print('Best value:', best)
print('Items taken:', chosen)
print('Weight used:', sum(w[item_names.index(c)] for c in chosen), 'of', W)

Best value: 35
Items taken: ['item2', 'item3', 'item4']
Weight used: 9 of 10


In [20]:
# Small example where the difference between 0/1 and unbounded is obvious.
# Unbounded picks Globe twice for a value of 8. 0/1 can only use each item once.
W_small = 4
wt_small = [1, 2, 3]
vals_small = [1, 4, 6]
names_small = ['Turtle', 'Globe', 'WaterMelon']

print('unbounded:', unboundedKnapsack(W_small, len(vals_small), wt_small, vals_small,
                                      [[nm] for nm in names_small]))
print('0/1      :', knapsack01(W_small, len(vals_small), wt_small, vals_small, names_small)[:2])

# Degenerate cases
print('zero capacity :', knapsack01(0, 3, wt_small, vals_small, names_small)[:2])
print('no items      :', knapsack01(10, 0, [], [], [])[:2])
print('nothing fits  :', knapsack01(1, 2, [5, 7], [50, 70], ['a', 'b'])[:2])

unbounded: (8, ['Globe', 'Globe'])
0/1      : (7, ['Turtle', 'WaterMelon'])
zero capacity : (0, [])
no items      : (0, [])
nothing fits  : (0, [])


In [21]:
# Large input case: 500 items, capacity 2000, so the table has about 1 million cells
import random

random.seed(7)
N, BIG_W = 500, 2000
big_w = [random.randint(1, 60) for _ in range(N)]
big_v = [random.randint(1, 120) for _ in range(N)]
big_names = ['i{}'.format(i) for i in range(N)]

t0 = time.perf_counter()
best, chosen, _ = knapsack01(BIG_W, N, big_w, big_v, big_names)
t1 = time.perf_counter()

print('Best value  :', best)
print('Items taken :', len(chosen))
print('Weight used :', sum(big_w[int(c[1:])] for c in chosen), 'of', BIG_W)
print('Time        : {:.3f} s for n*W = {} cells'.format(t1 - t0, N * BIG_W))

Best value  : 12262
Items taken : 157
Weight used : 2000 of 2000
Time        : 0.496 s for n*W = 1000000 cells


In [22]:
# Space-optimised 0/1 knapsack: one row, capacity loop backwards.
# K[x] plays two roles during pass j: cells at x or above are already row j,
# cells below x are still row j-1. Going backwards means K[x - wt[j]] is always
# the row j-1 value, which is what the 2D recurrence reads.

def knapsack01_1d(W, n, wt, vals):
    K = [0] * (W + 1)
    for j in range(n):
        for x in range(W, wt[j] - 1, -1):          # backwards
            K[x] = max(K[x], K[x - wt[j]] + vals[j])
    return K[W]

def knapsack01_1d_forwards(W, n, wt, vals):
    K = [0] * (W + 1)
    for j in range(n):
        for x in range(wt[j], W + 1):               # forwards: K[x - wt[j]] already holds item j
            K[x] = max(K[x], K[x - wt[j]] + vals[j])
    return K[W]

# classroom case: W = 10, values [20, 8, 14, 13, 35], weights [6, 2, 4, 3, 11]
W, wt, vals = 10, [6, 2, 4, 3, 11], [20, 8, 14, 13, 35]
names = ['item1', 'item2', 'item3', 'item4', 'item5']

print("2D 0/1 table    :", knapsack01(W, len(wt), wt, vals, names)[0])
print("1D backwards    :", knapsack01_1d(W, len(wt), wt, vals))
print("1D forwards     :", knapsack01_1d_forwards(W, len(wt), wt, vals), " <- items repeated, this is unbounded knapsack")

2D 0/1 table    : 35
1D backwards    : 35
1D forwards     : 42  <- items repeated, this is unbounded knapsack


### <font color='red'> Activity 4: Given coins of different denominations and a total amount, find the minimum number of coins needed to make up that amount. Implement both a top-down memoized solution and a bottom-up tabulation version, and handle the case where the amount cannot be reached. </font>

In [23]:
def coins_topdown(coins, amount):
    """Top-down memoized. solve(rem) is the fewest coins needed to make rem."""
    memo = {}

    def solve(rem):
        if rem == 0:
            return 0
        if rem < 0:
            return math.inf       # this branch overshot, so it is not a valid answer
        if rem in memo:
            return memo[rem]

        best = math.inf
        for c in coins:
            best = min(best, solve(rem - c) + 1)

        memo[rem] = best
        return best

    result = solve(amount)
    return -1 if result == math.inf else result


def coins_bottomup(coins, amount):
    """Bottom-up tabulation. Also records which coin was used so the change can be listed."""
    K = [math.inf] * (amount + 1)
    K[0] = 0
    pick = [None] * (amount + 1)

    for x in range(1, amount + 1):
        for c in coins:
            if c <= x and K[x - c] + 1 < K[x]:
                K[x] = K[x - c] + 1
                pick[x] = c

    if K[amount] == math.inf:
        return -1, []             # amount cannot be made from these coins

    change = []
    x = amount
    while x > 0:
        change.append(pick[x])
        x -= pick[x]
    return K[amount], change

In [24]:
# What goes wrong if -1 is the sentinel inside the recurrence instead of infinity.
# Two failures, depending on how the recurrence is written.

def coins_minus1_min(coins, amount):
    # written as K[x] = 1 + min(K[x - c]), seeded with -1
    K = [-1] * (amount + 1)
    K[0] = 0
    for x in range(1, amount + 1):
        K[x] = 1 + min(K[x - c] for c in coins if c <= x) if any(c <= x for c in coins) else -1
    return K[amount]

def coins_minus1_lt(coins, amount):
    # written with the strict-less update from coins_bottomup, seeded with -1
    K = [-1] * (amount + 1)
    K[0] = 0
    for x in range(1, amount + 1):
        for c in coins:
            if c <= x and K[x - c] + 1 < K[x]:
                K[x] = K[x - c] + 1
    return K[amount]

print(f"{'coins':<12} {'amount':>6} | {'inf sentinel':>12} | {'-1 with min()':>13} | {'-1 with <':>9}")
print("-" * 62)
for coins, amount in [([2], 3), ([2, 5], 6), ([1, 3, 4], 6), ([5, 10], 3)]:
    print(f"{str(coins):<12} {amount:>6} | {coins_bottomup(coins, amount)[0]:>12} | {coins_minus1_min(coins, amount):>13} | {coins_minus1_lt(coins, amount):>9}")

coins        amount | inf sentinel | -1 with min() | -1 with <
--------------------------------------------------------------
[2]               3 |           -1 |             0 |        -1
[2, 5]            6 |            3 |             0 |        -1
[1, 3, 4]         6 |            2 |             2 |        -1
[5, 10]           3 |           -1 |            -1 |        -1


In [25]:
tests = [
    ([1, 2, 5], 11),      # 5 + 5 + 1
    ([2], 3),             # impossible
    ([1, 3, 4], 6),       # greedy would say 4 + 1 + 1, the answer is 3 + 3
    ([1, 2, 5], 0),       # nothing to make
    ([7, 11], 100),
    ([5, 10], 3),         # impossible again
]

for coins, amount in tests:
    td = coins_topdown(coins, amount)
    bu, change = coins_bottomup(coins, amount)
    print('coins={:<12} amount={:<5} top-down={:<4} bottom-up={:<4} change={}'.format(
        str(coins), amount, td, bu, change))

coins=[1, 2, 5]    amount=11    top-down=3    bottom-up=3    change=[1, 5, 5]
coins=[2]          amount=3     top-down=-1   bottom-up=-1   change=[]
coins=[1, 3, 4]    amount=6     top-down=2    bottom-up=2    change=[3, 3]
coins=[1, 2, 5]    amount=0     top-down=0    bottom-up=0    change=[]
coins=[7, 11]      amount=100   top-down=12   bottom-up=12   change=[7, 7, 7, 7, 7, 7, 7, 7, 11, 11, 11, 11]
coins=[5, 10]      amount=3     top-down=-1   bottom-up=-1   change=[]


***